# Công đoạn 04 — Chuẩn hóa dữ liệu F54
## 1. Mục tiêu
Minh họa giữ nguyên 1–5, StandardScaler và MinMaxScaler trên 800 mẫu Train với 54 câu hỏi. Chưa lựa chọn phương pháp, mô hình, đặc trưng hoặc đánh giá hiệu năng. Không dùng Test để quyết định.

## 2. Cơ sở lý thuyết
Likert là thang thứ bậc. Mã hóa cố định năm mức sang 1–5 và dùng scaler là lựa chọn kỹ thuật, không xác nhận khoảng cách tâm lý bằng nhau.

- Giữ nguyên: x' = x, miền lý thuyết [1,5].
- StandardScaler: x' = (x − μ_train)/σ_train, với phương sai ddof=0. Không có miền hữu hạn cố định; không bảo đảm phân phối chuẩn.
- MinMaxScaler: x' = (x − min_train)/(max_train − min_train), mặc định [0,1] trên Train. Mẫu mới có thể ngoài [0,1] vì mặc định clip=False.

Hai scaler nhạy với cực trị; chuẩn hóa không làm dữ liệu thứ bậc thành dữ liệu khoảng, không tự giảm dư thừa hay chứng minh tăng hiệu năng.

## 3. Tái lập dữ liệu
Đọc workbook; kiểm tra và giữ nguyên Likert văn bản, trim nhãn như Notebook 01. Group bằng tuple đủ 54 câu, factorize theo thứ tự xuất hiện; StratifiedGroupKFold 5 folds, shuffle=True, random_state=42, fold đầu tiên. Đối chiếu trực tiếp các phép gán tạo nhóm/chia của Notebook 02 để kiểm tra index tương đương. Không chạy thống kê Test của notebook đó.

In [1]:
from pathlib import Path
import sys, ast
import pandas as pd
import numpy as np
from IPython.display import display
from sklearn.model_selection import StratifiedGroupKFold
import nbformat
assert sys.version_info[:2] == (3,14)
ROOT = next(p for p in (Path.cwd(),*Path.cwd().parents) if (p/'data_AI_1000_rows.xlsx').is_file())
raw = pd.read_excel(ROOT/'data_AI_1000_rows.xlsx',sheet_name='AUGMENTED_RAW_1000')
metadata = ['Dấu thời gian','Tên Sinh viên','Ngành học','Giới tính']
assert raw.shape == (1000,58) and list(raw.columns[:4]) == metadata
questions = list(raw.columns[4:])
levels = ['Chưa bao giờ đúng','Chỉ đúng 1 vài trường hợp','Chỉ đúng 1 nửa','Gần như là đúng','Hoàn toàn đúng']
X = raw[questions].copy()
y = raw['Ngành học'].astype('string').str.strip()
assert X.notna().all().all() and X.isin(levels).all().all()
assert y.notna().all() and y.ne('').all() and y.nunique() == 33
assert X.shape[1] == 54 and not set(metadata).intersection(X.columns)
groups = pd.Series(pd.factorize(pd.Series(list(X.itertuples(index=False, name=None))), sort=False)[0], index=X.index)
splitter = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=42)
train_pos, test_pos = next(splitter.split(X, y, groups))
ref = nbformat.read(ROOT/'data/ml_pipeline/notebooks/02_chia_du_lieu.ipynb',4)
src = next(c.source for c in ref.cells if c.cell_type=='code' and 'from sklearn.model_selection import StratifiedGroupKFold' in c.source)
nodes = [node for node in ast.parse(src).body if isinstance(node,ast.Assign) and any(
    isinstance(t,ast.Name) and t.id in {'groups','splitter'} or isinstance(t,ast.Tuple) and [v.id for v in t.elts]==['train_pos','test_pos'] for t in node.targets)]
assert len(nodes)==3
check = {'pd':pd,'X':X,'y':y,'StratifiedGroupKFold':StratifiedGroupKFold}
exec(compile(ast.Module(body=nodes,type_ignores=[]),'<Notebook02 split>','exec'),check)
assert groups.equals(check['groups'])
assert np.array_equal(train_pos,check['train_pos']) and np.array_equal(test_pos,check['test_pos'])
X_train,y_train = X.iloc[train_pos].copy(),y.iloc[train_pos].copy()
assert X_train.shape==(800,54) and y_train.nunique()==33
assert len(test_pos)==200 and len(train_pos)+len(test_pos)==len(X)
assert X.index.is_unique and not set(train_pos).intersection(test_pos)
assert set(groups.iloc[train_pos]).isdisjoint(groups.iloc[test_pos])
print('PASS: index đúng Notebook 02; Train 800 x 54, 33 ngành; Test 200; không giao index/vector.')
del raw,X,y,groups,check,test_pos,ref


PASS: index đúng Notebook 02; Train 800 x 54, 33 ngành; Test 200; không giao index/vector.


## 4. Kiểm tra Train trước chuẩn hóa
Ánh xạ theo thứ tự năm nhãn đã xác nhận, không học tham số. Bảng dùng SD ddof=0 để thống nhất với StandardScaler; q1–q54 giữ nguyên thứ tự workbook và có bảng đối chiếu nội dung.

In [2]:
from sklearn.preprocessing import StandardScaler, MinMaxScaler
mapping = dict(zip(levels, range(1,6)))
X_train_numeric = X_train.apply(lambda col: col.map(mapping)).astype('int64')
X_train_numeric.columns = [f'q{i}' for i in range(1,55)]
assert X_train_numeric.shape == (800,54)
assert X_train_numeric.notna().all().all()
assert all(pd.api.types.is_numeric_dtype(t) for t in X_train_numeric.dtypes)
assert X_train_numeric.isin(range(1,6)).all().all()
assert X_train_numeric.index.equals(X_train.index)
assert not set(metadata).intersection(X_train_numeric.columns)
display(pd.DataFrame({'Câu':X_train_numeric.columns,'Nội dung workbook':questions}))
before = pd.DataFrame({'min':X_train_numeric.min(), 'max':X_train_numeric.max(),
    'mean':X_train_numeric.mean(), 'std (ddof=0)':X_train_numeric.std(ddof=0),
    'variance (ddof=0)':X_train_numeric.var(ddof=0)})
zero_variance = before.index[before['variance (ddof=0)'].eq(0)].tolist()
display(before.round(6))
print('Đặc trưng phương sai bằng 0:',zero_variance)
original_snapshot = X_train_numeric.copy(deep=True)
# Phương án A: bản sao dữ liệu số nguyên ban đầu.
X_train_original = X_train_numeric.copy()


Đặc trưng phương sai bằng 0: []


    Câu                                                              Nội dung workbook
0    q1                                                          1. Tôi có tính tự lập
1    q2                                                        2. Tôi suy nghĩ thực tế
2    q3                                  3. Tôi là người thích nghi với môi trường mới
3    q4                        4. Tôi có thể vận hành, điều khiển các máy móc thiết bị
4    q5                   5. Tôi làm các công việc thủ công như gấp giấy, đan, móc,...
5    q6                          6. Tôi thích tiếp xúc với thiên nhiên động vật cây cỏ
6    q7                    7. Tôi thích những công việc sử dụng tay chân hơn là trí óc
7    q8                                8. Tôi thích những công việc  thấy ngay kết quả
8    q9                 9. Tôi thích những công việc ngoài trời hơn là trong văn phòng
9   q10                                  10. Tôi có tìm hiểu khám phá nhiều vấn đề mới
10  q11                                    

     min  max     mean  std (ddof=0)  variance (ddof=0)
q1     1    5  3.46750      1.031476           1.063944
q2     1    5  3.78375      0.980809           0.961986
q3     1    5  3.66000      1.025622           1.051900
q4     1    5  2.69750      1.036095           1.073494
q5     1    5  2.56000      1.164646           1.356400
q6     1    5  3.91000      1.092886           1.194400
q7     1    5  2.45125      1.057177           1.117623
q8     1    5  3.27000      1.144378           1.309600
q9     1    5  2.82000      1.119196           1.252600
q10    1    5  3.38250      1.086827           1.181194
q11    1    5  3.15375      1.010005           1.020111
q12    1    5  3.36375      1.020508           1.041436
q13    1    5  3.00750      1.242958           1.544944
q14    1    5  3.09000      1.088301           1.184400
q15    1    5  3.10500      1.230437           1.513975
q16    1    5  3.54375      1.087698           1.183086
q17    1    5  2.65250      1.145314           1

## 5. StandardScaler
Fit đúng 800 mẫu Train, chỉ 54 câu. Nếu có cột hằng, StandardScaler đưa cột đó về 0; kiểm tra SD≈1 chỉ áp dụng cho cột không hằng.

In [3]:
standard_scaler = StandardScaler()
standard_scaler.fit(X_train_numeric)
X_train_standard = pd.DataFrame(standard_scaler.transform(X_train_numeric),
    index=X_train_numeric.index, columns=X_train_numeric.columns)
nonconstant = X_train_numeric.var(ddof=0).gt(0)
assert standard_scaler.n_samples_seen_ == 800
assert standard_scaler.n_features_in_ == 54
assert np.array_equal(standard_scaler.feature_names_in_, X_train_numeric.columns)
assert np.allclose(standard_scaler.mean_, X_train_numeric.mean())
assert np.allclose(standard_scaler.var_, X_train_numeric.var(ddof=0))
assert np.allclose(X_train_standard.mean(),0,atol=1e-12)
assert np.allclose(X_train_standard.std(ddof=0)[nonconstant],1,atol=1e-12)
standard_stats = pd.DataFrame({'min':X_train_standard.min(),'max':X_train_standard.max(),
    'mean':X_train_standard.mean(),'std (ddof=0)':X_train_standard.std(ddof=0)})
display(standard_stats.round(6))
print('Sai lệch mean lớn nhất:',X_train_standard.mean().abs().max())
print('Sai lệch SD so với 1 lớn nhất:',(X_train_standard.std(ddof=0)[nonconstant]-1).abs().max())

Sai lệch mean lớn nhất: 4.618527782440651e-16
Sai lệch SD so với 1 lớn nhất: 2.220446049250313e-16


          min       max  mean  std (ddof=0)
q1  -2.392202  1.485734   0.0           1.0
q2  -2.838219  1.240048   0.0           1.0
q3  -2.593549  1.306525  -0.0           1.0
q4  -1.638363  2.222286   0.0           1.0
q5  -1.339463  2.095057  -0.0           1.0
q6  -2.662675  0.997359  -0.0           1.0
q7  -1.372760  2.410902   0.0           1.0
q8  -1.983611  1.511739  -0.0           1.0
q9  -1.626167  1.947827   0.0           1.0
q10 -2.192160  1.488277   0.0           1.0
q11 -2.132414  1.827960  -0.0           1.0
q12 -2.316249  1.603369  -0.0           1.0
q13 -1.615099  1.603031   0.0           1.0
q14 -1.920424  1.755028   0.0           1.0
q15 -1.710774  1.540103   0.0           1.0
q16 -2.338656  1.338837  -0.0           1.0
q17 -1.442836  2.049657   0.0           1.0
q18 -2.015178  1.990145  -0.0           1.0
q19 -2.309012  1.040416   0.0           1.0
q20 -2.303435  1.225044  -0.0           1.0
q21 -2.921819  0.941743  -0.0           1.0
q22 -1.480866  1.983149   0.0   

## 6. MinMaxScaler
Fit chỉ trên Train với feature_range=(0,1); cột không hằng đạt min=0, max=1. Cột hằng nếu có sẽ thành 0. Không transform Test.

In [4]:
minmax_scaler = MinMaxScaler(feature_range=(0,1))
minmax_scaler.fit(X_train_numeric)
X_train_minmax = pd.DataFrame(minmax_scaler.transform(X_train_numeric),
    index=X_train_numeric.index,columns=X_train_numeric.columns)
assert minmax_scaler.n_samples_seen_ == 800
assert minmax_scaler.n_features_in_ == 54
assert np.array_equal(minmax_scaler.feature_names_in_,X_train_numeric.columns)
assert np.allclose(minmax_scaler.data_min_,X_train_numeric.min())
assert np.allclose(minmax_scaler.data_max_,X_train_numeric.max())
assert np.allclose(X_train_minmax.min(),0,atol=1e-12)
assert np.allclose(X_train_minmax.max()[nonconstant],1,atol=1e-12)
assert X_train_minmax.ge(-1e-12).all().all() and X_train_minmax.le(1+1e-12).all().all()
minmax_stats = pd.DataFrame({'min':X_train_minmax.min(),'max':X_train_minmax.max(),
    'mean':X_train_minmax.mean(),'std (ddof=0)':X_train_minmax.std(ddof=0)})
display(minmax_stats.round(6))

     min  max      mean  std (ddof=0)
q1   0.0  1.0  0.616875      0.257869
q2   0.0  1.0  0.695938      0.245202
q3   0.0  1.0  0.665000      0.256405
q4   0.0  1.0  0.424375      0.259024
q5   0.0  1.0  0.390000      0.291161
q6   0.0  1.0  0.727500      0.273222
q7   0.0  1.0  0.362812      0.264294
q8   0.0  1.0  0.567500      0.286094
q9   0.0  1.0  0.455000      0.279799
q10  0.0  1.0  0.595625      0.271707
q11  0.0  1.0  0.538438      0.252501
q12  0.0  1.0  0.590938      0.255127
q13  0.0  1.0  0.501875      0.310739
q14  0.0  1.0  0.522500      0.272075
q15  0.0  1.0  0.526250      0.307609
q16  0.0  1.0  0.635938      0.271924
q17  0.0  1.0  0.413125      0.286328
q18  0.0  1.0  0.503125      0.249668
q19  0.0  1.0  0.689375      0.298558
q20  0.0  1.0  0.652812      0.283408
q21  0.0  1.0  0.756250      0.258828
q22  0.0  1.0  0.427500      0.288682
q23  0.0  1.0  0.439375      0.306563
q24  0.0  1.0  0.388750      0.321080
q25  0.0  1.0  0.461875      0.296516
q26  0.0  1.

## 7. Bảng so sánh thực tế
Khoảng mean và SD dưới đây là min–max giữa 54 đặc trưng, không phải thống kê gộp mọi ô. Không dùng Accuracy hoặc Test để chọn phương án.

In [5]:
rows = []
for name, data in [('Giữ nguyên 1–5',X_train_original),
                   ('StandardScaler',X_train_standard),('MinMaxScaler',X_train_minmax)]:
    rows.append({'Phương án':name,'Min toàn bộ':data.min().min(),'Max toàn bộ':data.max().max(),
        'Mean nhỏ nhất':data.mean().min(),'Mean lớn nhất':data.mean().max(),
        'SD nhỏ nhất (ddof=0)':data.std(ddof=0).min(),
        'SD lớn nhất (ddof=0)':data.std(ddof=0).max()})
comparison = pd.DataFrame(rows).set_index('Phương án')
display(comparison.round(6))
assert X_train_numeric.equals(original_snapshot)
assert X_train_original.equals(original_snapshot)
assert X_train_standard.index.equals(X_train_numeric.index)
assert X_train_minmax.index.equals(X_train_numeric.index)
assert X_train_standard.columns.equals(X_train_numeric.columns)
assert X_train_minmax.columns.equals(X_train_numeric.columns)
print('PASS: giữ nguyên dữ liệu đầu vào, thứ tự 54 câu và index; scaler fit 800 mẫu Train; không transform Test.')

PASS: giữ nguyên dữ liệu đầu vào, thứ tự 54 câu và index; scaler fit 800 mẫu Train; không transform Test.


                Min toàn bộ  Max toàn bộ  Mean nhỏ nhất  Mean lớn nhất  SD nhỏ nhất (ddof=0)  SD lớn nhất (ddof=0)
Phương án                                                                                                         
Giữ nguyên 1–5     1.000000     5.000000       2.451250        4.31500              0.820533              1.284319
StandardScaler    -4.040055     2.410902      -0.000000        0.00000              1.000000              1.000000
MinMaxScaler       0.000000     1.000000       0.362812        0.82875              0.205133              0.321080

| Phương án | Nguyên lý/khoảng | Ưu điểm | Hạn chế và bối cảnh thuật toán |
|---|---|---|---|
| Giữ nguyên | x; [1,5] | Dễ giải thích, các câu vốn cùng thang | Khác SD giữa câu vẫn có thể ảnh hưởng khoảng cách; cây quyết định và mô hình tổ hợp cây thường không cần scaling |
| StandardScaler | (x−μ)/σ; không giới hạn cố định | Đưa các biến không hằng về mean≈0, SD≈1 | Nhạy cực trị; có thể thay đổi trọng số tương đối giữa câu; thường cân nhắc cho SVM, kNN, mô hình tuyến tính có regularization, mạng neural |
| MinMaxScaler | (x−min)/(max−min); Train [0,1] | Miền dễ hiểu, giữ thứ tự | Nhạy cực trị; mẫu mới có thể ngoài [0,1]; thường cân nhắc cho thuật toán dùng khoảng cách hoặc mạng neural |

## 8. Nhận xét
Các câu vốn cùng miền 1–5 nên scaling không mặc nhiên cần thiết. StandardScaler thay đổi mức đóng góp tương đối theo SD của từng câu; MinMax dùng cực trị Train. Hai biến đổi đều giữ thứ tự trong từng câu không hằng. Không có kết quả mô hình để kết luận phương án nào tốt hơn. Các bảng chỉ minh họa phép biến đổi, không lựa chọn scaler.

## 9. Kết luận và chống leakage trong cross-validation
**Các ma trận chuẩn hóa ở notebook này chỉ để minh họa. Không đưa trực tiếp chúng vào Cross-Validation ở Notebook 06.** Nếu dùng scaler fit toàn Train trước CV, các validation folds đã ảnh hưởng mean/SD hoặc min/max, gây leakage nội bộ.

Khi huấn luyện/tuning, truyền dữ liệu Train chưa fit scaler vào sklearn Pipeline để scaler được fit lại riêng trên training portion của từng fold. Mọi feature selection học từ dữ liệu cũng phải nằm trong Pipeline. CV cần tiếp tục giữ các vector trùng cùng nhóm, không để chúng đi qua ranh giới training/validation. Không đưa mô hình vào notebook này.

Sau khi chọn cấu hình bằng CV trên Train, mới fit Pipeline cuối cùng trên toàn Train và đánh giá **một lần trên Test**. Không fit Test hoặc dùng Test quyết định preprocessing.

Code notebook này chỉ fit/transform X_train_numeric; không đưa y hay metadata vào scaler, không fit toàn bộ 1.000 mẫu và không transform Test. Không chọn đặc trưng, PCA hoặc huấn luyện; không lưu scaler. Dừng ở Notebook 04.

Kiểm chứng: thực thi tuần tự toàn bộ code cell trong tiến trình Python 3.14 mới, lưu bảng/outputs trực tiếp trong notebook. Dùng phương án này theo giới hạn quyền Windows của Jupyter đã ghi nhận ở Notebook 02; chưa kiểm chứng bằng kernel Jupyter và không thử tạo môi trường hoặc file phụ.
